# iSLAT Animated Population Diagram

This notebook animates the population (rotation) diagram of one molecule while one of its
parameters steps through an array of values. By default it steps **CO** through temperatures
from 300 K to 3000 K.

Each frame is drawn by iSLAT's own `PopulationDiagramPlot`, the same plot as the GUI's
population diagram: $\ln\!\bigl(4\pi F / (h\nu\, A_u\, g_u)\bigr)$ against $E_u$ (K) for every
line in the line list. The axis limits stay the same for the whole animation, so you see the
points move rather than the axes.

A straight-line fit to the stronger lines is drawn on each frame. For optically thin lines in
LTE its slope is $-1/T_\mathrm{rot}$, so the fitted $T_\mathrm{rot}$ follows the model
temperature. Where the lines become optically thick, the diagram curves and the fit falls
away from the model temperature.

**To use it:** edit the configuration cell (section 2) and run all cells.
- `MOLECULE_CONFIG` is passed straight to `Molecule(...)`: line list, temperature, column
  density, radius, distance, and so on.
- `PARAMETER` is any `Molecule` attribute: `"temp"`, `"n_mol"`, `"radius"`, `"distance"`,
  `"broad"`, ...
- `VALUES` is the array of values for that parameter, one frame each.

## 1. Setup

In [1]:
%matplotlib inline

import contextlib
import io
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FFMpegWriter, FuncAnimation, PillowWriter
from IPython.display import HTML, display

from iSLAT.Modules.DataTypes import Molecule
from iSLAT.Modules.FileHandling import hitran_data_folder_path
from iSLAT.Modules.Plotting import PopulationDiagramPlot, load_theme
from iSLAT.Modules.DataTypes.PlotAxisRegistry import PlotAxisRegistry

import iSLAT.Modules.DataTypes.HITRANQuantumSchemas

print("Imports successful!")

Imports successful!


## 2. Configuration

- `MOLECULE_CONFIG`: the molecule, as keyword arguments to `Molecule(...)`. Use any line list
  in `hitran_data_folder_path` or a path of your own.
- `PARAMETER` / `VALUES`: the parameter to step through and its values. The default steps
  CO from 300 K to 3000 K in 100 K steps. For a column density, logarithmic steps work best,
  e.g. `PARAMETER = "n_mol"` with `VALUES = np.logspace(15, 20, 26)`.
- `X_AXIS` / `Y_AXIS`: the line properties on the two axes, any of the axes the GUI's
  *Plot Settings* offers for the molecule (section 3 lists their keys). The default is the
  population diagram, `"eu"` ($E_\mathrm{up}$) against `"rd_yax"`
  ($\ln(4\pi F / h\nu A g)$); for example `X_AXIS = "wavelength"`, `Y_AXIS = "tau"` with
  `Y_LOG = True` shows the optical depth of each line across the spectrum. Categorical
  properties (`"vib_band"`, `"lev_up"`, `"qn_upper:Br"`, ...) give one position per category.
  - `X_LOG` / `Y_LOG`: a logarithmic axis (ignored for a categorical one).
- `COLOR_BY`: `None` for the GUI's plain points, or a property to color the points by
  (`"a_stein"`, `"wavelength"`, `"intens"`, `"e_low"`, `"lev_up"`, ...; see the
  Population_Diagram_Examples notebook). Quantum numbers are `"qn_upper:FIELD"` /
  `"qn_lower:FIELD"`, e.g. `"qn_upper:v"` (vibrational) or `"qn_lower:J"` (rotational, CO);
  they need the `HITRANQuantumSchemas` import in section 1.
- `COLOR_ALL_LINES`: with `COLOR_BY`, color every line of the line list, as the plain
  diagram shows them. `False` draws only the stronger lines (flux above 1 % of the strongest
  line's), as the GUI does when coloring. With `True` the color scale also spans every line,
  so a value keeps its color from frame to frame.
- `SIZE_BY`: a property that sets each point's marker size, or `None` for one size
  (`MARKER_SIZE`). The default `"tau"` is the line-center optical depth (opacity), so the
  optically thick lines stand out as large points. Any `COLOR_BY` property works.
  - `SIZE_LOG`: scale the sizes with the log of the value (suits `"tau"`, which spans many
    decades).
  - `SIZE_LIMITS`: the values that get the smallest and largest marker; values beyond them
    are clipped. Fixed limits keep a size meaning the same value in every frame. The default
    `(1e-2, 1e2)` runs from optically thin ($\tau \le 0.01$) to very thick
    ($\tau \ge 100$). `None` uses each frame's own range.
  - `SIZE_RANGE`: the smallest and largest marker area, in points$^2$.
- `SHOW_FIT`: draw the straight-line fit and its $T_\mathrm{rot}$ on each frame (only on
  the default, linear `"eu"` / `"rd_yax"` axes, where the fit means a temperature).
- `FIXED_LIMITS`: keep the same axis limits on every frame (the union of every frame's own
  limits). With `False`, each frame uses its own limits, as the GUI would.
- `INLINE_DPI` / `INLINE_MAX_MB`: the resolution and size limit of the animation shown in
  the notebook (the saved file uses `SAVE_DPI`). At 100 dpi each frame is about 100 kB, so a
  long sweep such as 271 temperatures is about 27 MB, past matplotlib's 20 MB default, which
  drops the frames after it. The animation is stored in the notebook when it is saved, so a
  lower `INLINE_DPI` or fewer `VALUES` keeps the file smaller.
- `THEME`: the plot colors, as an iSLAT theme. The default, `load_theme("DarkTheme")`,
  draws on a dark background, where bright point colors stand out more than on white; the
  theme also colors the axes, labels, legends, colorbar, the parameter box and the fit line.
  Other bundled themes are `"LightTheme"` and `"PastelBlue"`, and `None` is iSLAT's default
  light style.
- `SAVE_PATH`: where to save the animation (`.gif`; `.mp4` needs ffmpeg), or `None` to only
  show it here.

In [2]:
MOLECULE_CONFIG = dict(
    name="CO",
    filepath=str(hitran_data_folder_path / "data_Hitran_CO.par"),
    displaylabel="CO",
    temp=600.0,           # K
    n_mol=1e18,            # cm^-2
    radius=0.5,            # au
    distance=160.0,        # pc
    wavelength_range=(4.5, 28.0),  # um
    fwhm=130.0,            # km/s
    broad=1.0,             # km/s
)

#PARAMETER = "temp"
#VALUES = np.arange(300.0, 3001.0, 100.0) # K
PARAMETER = "n_mol"
VALUES = np.logspace(15, 25, 50)  # cm^-2


#COLOR_BY = None            # e.g. "a_stein", "wavelength", "e_low"
#COLOR_BY = "qn_upper:v"
COLOR_BY = "eu"
COLOR_MAP = "viridis"         # 12 light colors, one per CO band; stands out on the dark theme
COLOR_ALL_LINES = True     # with COLOR_BY: every line, not only the strong ones
MARKER_SIZE = 6            # points^2; the GUI uses 0.5 for a single molecule (used when SIZE_BY is None)
SIZE_BY = "intens"            # marker size from a line property: "tau" = optical depth (opacity); None for one size
SIZE_LOG = True            # size from log10 of the value
SIZE_LIMITS = (5e-2, 5e2)  # values given the smallest / largest marker; None = each frame's own range
SIZE_RANGE = (1.5, 40.0)   # smallest / largest marker area, points^2
SHOW_FIT = False
FIXED_LIMITS = True
X_AXIS = "velocity_sep"              # any key section 3 lists, as in the GUI's Plot Settings; "eu" = E_up (K)
Y_AXIS = "rd_yax"          # "rd_yax" = the population diagram's ln(4 pi F / (h nu A g))
X_LOG = False              # log scale (ignored for a categorical axis such as "vib_band")
Y_LOG = False

FIGSIZE = (13, 7)
FRAME_INTERVAL_MS = 300    # time between frames
INLINE_DPI = 100           # resolution of the animation shown in the notebook
INLINE_MAX_MB = 200        # size limit of the animation shown here; matplotlib drops frames past it
THEME = load_theme("DarkTheme")  # "LightTheme", "PastelBlue", or None for the default light style

SAVE_PATH = None           # e.g. Path("CO_population_diagram_temp.gif")
SAVE_FPS = 30
SAVE_DPI = 200

## 3. Create the molecule

Also lists the axes `X_AXIS` / `Y_AXIS` can use for this molecule: the list the GUI's
*Plot Settings* offers (the quantum numbers depend on the molecule's line list).

In [3]:
molecule = Molecule(**MOLECULE_CONFIG)

if not hasattr(molecule, PARAMETER):
    raise AttributeError(f"Molecule has no parameter {PARAMETER!r}")

print(f"{molecule.name}: {len(VALUES)} frames of {PARAMETER} from {VALUES[0]:g} to {VALUES[-1]:g}")
print(f"Starting value: {PARAMETER} = {getattr(molecule, PARAMETER)!r}")

# The axes the GUI's Plot Settings offers for this molecule: key -> name shown in the GUI.
AXIS_OPTIONS = {
    entry.key: entry.display_name
    for entry in PlotAxisRegistry.get_property_options(
        species=PopulationDiagramPlot(molecule=molecule).get_molecule_ids())
}
for name, key in (("X_AXIS", X_AXIS), ("Y_AXIS", Y_AXIS)):
    if PlotAxisRegistry.canonical_key(key) not in AXIS_OPTIONS:
        raise ValueError(f"{name} = {key!r} is not an axis for {molecule.name}; use one of {list(AXIS_OPTIONS)}")

print(f"\nAxes: X = {AXIS_OPTIONS[PlotAxisRegistry.canonical_key(X_AXIS)]}, "
      f"Y = {AXIS_OPTIONS[PlotAxisRegistry.canonical_key(Y_AXIS)]}. Every choice (key: name):")
for key, display_name in AXIS_OPTIONS.items():
    print(f"  {key:24s} {display_name}")

CO: 50 frames of n_mol from 1e+15 to 1e+25
Starting value: n_mol = 1e+18

Axes: X = Velocity sep. (km/s), Y = Pop. diagram Y [ln(4πF/hνAg)]. Every choice (key: name):
  wavelength               Wavelength (µm)
  eu                       E_up (K)
  e_low                    E_low (K)
  a_stein                  Einstein A (s⁻¹)
  g_up                     g_up
  g_low                    g_low
  freq                     Frequency (Hz)
  rd_yax                   Pop. diagram Y [ln(4πF/hνAg)]
  intens                   Line intensity
  tau                      Optical depth (τ)
  fwhm_instrumental_kms    Instrumental FWHM (km/s)
  fwhm_convolved_kms       Convolved FWHM (km/s)
  velocity_sep             Velocity sep. (km/s)
  lev_up                   Upper level (lev_up)
  lev_low                  Lower level (lev_low)
  vib_band                 Vib. band
  molecule                 Molecule (species)
  qn_upper:v               v (upper)
  qn_upper:Br              Br (upper)
  qn_upper:J      

## 4. Animation helpers

- `set_parameter` sets the parameter and recalculates the intensities, without the
  "Calculating intensity" message the molecule prints for every frame.
- `rotational_fit` fits a straight line to the lines that `PopulationDiagramPlot` counts as
  valid (flux above 1 % of the strongest line) and returns the slope, the intercept,
  $T_\mathrm{rot} = -1/\mathrm{slope}$ and the $E_u$ range of those lines. The dashed line
  is drawn over that range only.
- `common_limits` draws every frame once (on the chosen axes), without showing it, and returns the smallest axis
  limits that hold all of them.
- `make_population_diagram_animation` builds the `FuncAnimation`. Each frame sets the
  parameter, redraws the diagram with `PopulationDiagramPlot.generate_plot()` and adds the
  parameter value and, on the default axes, the fit. With `size_by` the marker sizes follow that property
  (`PopulationDiagramPlot.size_by()`), with a legend of example sizes in the lower left.
  The plot is given the figure as well as the axes, so the theme colors the whole figure on
  every frame; the parameter box and the fit line take their colors from the theme too.

In [4]:
# Label and value format for the common parameters; others use their own name.
PARAMETER_LABELS = {
    "temp":     ("$T$", "K", "{:.0f}"),
    "n_mol":    ("$N$", "cm$^{-2}$", "{:.2e}"),
    "radius":   ("$R$", "au", "{:.3g}"),
    "distance": ("$d$", "pc", "{:.4g}"),
    "broad":    ("$\\Delta v$", "km/s", "{:.3g}"),
}


def parameter_text(parameter, value):
    label, unit, fmt = PARAMETER_LABELS.get(parameter, (parameter, "", "{:.4g}"))
    return f"{label} = {fmt.format(value)} {unit}".strip()


def set_parameter(molecule, parameter, value):
    # Set one Molecule parameter and recalculate its intensities quietly.
    setattr(molecule, parameter, value)
    with contextlib.redirect_stdout(io.StringIO()):
        molecule.calculate_intensity()


def rotational_fit(pdp):
    # Straight-line fit of ln(4 pi F / (h nu A g)) against E_u over the valid lines.
    if not pdp.component_data:
        return None
    data = pdp.component_data[0]
    eu, rd, valid = data["eu"], data["rd_yax"], data["valid_mask"]
    keep = valid & np.isfinite(eu) & np.isfinite(rd)
    if keep.sum() < 3:
        return None
    slope, intercept = np.polyfit(eu[keep], rd[keep], 1)
    t_rot = -1.0 / slope if slope < 0 else np.inf
    eu_range = (eu[keep].min(), eu[keep].max())
    return slope, intercept, t_rot, eu_range


def common_limits(pdp, molecule, parameter, values, axes):
    # Draw every frame once (not shown) and return the x / y limits that hold them all.
    # set_axes sets the axis properties too, so the chosen `axes` go with every call.
    pdp.set_axes(**axes, x_lim=None, y_lim=None, regenerate=False)
    xlims, ylims = [], []
    for value in values:
        set_parameter(molecule, parameter, value)
        pdp.generate_plot()
        xlims.append(pdp.ax.get_xlim())
        ylims.append(pdp.ax.get_ylim())
    xlims, ylims = np.array(xlims), np.array(ylims)
    return (xlims[:, 0].min(), xlims[:, 1].max()), (ylims[:, 0].min(), ylims[:, 1].max())


def make_population_diagram_animation(
    molecule, parameter, values, *,
    color_by=None, cmap="viridis", color_all_lines=True, marker_size=6,
    size_by=None, size_log=True, size_limits=None, size_range=(1.5, 40.0),
    show_fit=True, fixed_limits=True,
    x_axis="eu", y_axis="rd_yax", x_log=False, y_log=False,
    figsize=(7, 5), interval=200, theme=None,
):
    # Return (animation, figure) for the population diagram of `molecule` over `values`.
    fig, ax = plt.subplots(figsize=figsize, layout="constrained")
    # With fig= the plot applies the theme to the whole figure (background, axes, labels,
    # legends, colorbar) each time it draws; with only ax= it would leave the figure alone.
    pdp = PopulationDiagramPlot(molecule=molecule, ax=ax, fig=fig, theme=theme)
    axes = dict(x_prop=x_axis, y_prop=y_axis, x_log=x_log, y_log=y_log)
    pdp.set_axes(**axes, regenerate=False)
    pdp.set_marker_size(marker_size, regenerate=False)
    if color_by is not None:
        pdp.color_by(color_by, cmap=cmap, all_lines=color_all_lines, regenerate=False)
    if size_by is not None:
        size_min, size_max = size_limits if size_limits is not None else (None, None)
        pdp.size_by(size_by, min_size=size_range[0], max_size=size_range[1],
                    vmin=size_min, vmax=size_max, log_scale=size_log, regenerate=False)

    if fixed_limits:
        xlim, ylim = common_limits(pdp, molecule, parameter, values, axes)
        pdp.set_axes(**axes, x_lim=("exact", *xlim), y_lim=("exact", *ylim), regenerate=False)

    # The rotational fit is a straight line in E_up against ln(4 pi F / (h nu A g)) only.
    canonical = PlotAxisRegistry.canonical_key
    fit_axes = (canonical(x_axis), canonical(y_axis)) == ("eu", "rd_yax") and not (x_log or y_log)

    # Colors of the parameter box and the fit line: from the theme, or the light defaults.
    th = theme or {}
    fg = th.get("foreground", "black")
    box = dict(boxstyle="round", facecolor=th.get("background_accent_color", "white"),
               edgecolor=th.get("foreground", "0.6"), alpha=0.85)
    fit_color = th.get("fit_line_color", "tab:red")

    def draw_frame(index):
        value = values[index]
        set_parameter(molecule, parameter, value)
        pdp.generate_plot()  # clears the axes and draws this frame's diagram
        lines = [parameter_text(parameter, value)]
        fit = rotational_fit(pdp) if show_fit and fit_axes else None
        if fit is not None:
            slope, intercept, t_rot, eu_range = fit
            x = np.array(eu_range)  # only over the lines that were fitted
            ax.plot(x, slope * x + intercept, ls="--", lw=1.5, color=fit_color)
            lines.append(f"$T_\\mathrm{{rot}}$ (fit) = {t_rot:.0f} K")
        # Top center, clear of the legends (category colors upper right, sizes lower left).
        ax.text(0.5, 0.97, "\n".join(lines), transform=ax.transAxes, ha="center", va="top",
                multialignment="center", fontsize=11, color=fg, bbox=box)
        ax.set_title(f"{pdp.ax.get_title()}  ({index + 1}/{len(values)})", fontsize="medium", color=fg)
        return []

    anim = FuncAnimation(fig, draw_frame, frames=len(values), interval=interval, blit=False)
    return anim, fig

## 5. Make the animation

Building the animation draws every frame (twice with `FIXED_LIMITS`, once to find the
limits), which takes a few seconds for CO. Large line lists such as H$_2$O take longer per
frame. The molecule is put back to its starting value afterwards.

In [5]:
start_value = getattr(molecule, PARAMETER)

anim, fig = make_population_diagram_animation(
    molecule, PARAMETER, VALUES,
    color_by=COLOR_BY, cmap=COLOR_MAP, color_all_lines=COLOR_ALL_LINES, marker_size=MARKER_SIZE,
    size_by=SIZE_BY, size_log=SIZE_LOG, size_limits=SIZE_LIMITS, size_range=SIZE_RANGE,
    show_fit=SHOW_FIT, fixed_limits=FIXED_LIMITS,
    x_axis=X_AXIS, y_axis=Y_AXIS, x_log=X_LOG, y_log=Y_LOG,
    figsize=FIGSIZE, interval=FRAME_INTERVAL_MS, theme=THEME,
)
# iSLAT sets matplotlib's savefig.dpi to 300 for exports; the inline frames use INLINE_DPI.
# animation.embed_limit: matplotlib's 20 MB default drops the later frames of long sweeps.
with plt.rc_context({"savefig.dpi": INLINE_DPI, "animation.embed_limit": INLINE_MAX_MB}):
    html = anim.to_jshtml()
plt.close(fig)  # show only the animation, not a static copy of the last frame

set_parameter(molecule, PARAMETER, start_value)
display(HTML(html))

## 6. Save the animation

Set `SAVE_PATH` in section 2 (or here) and run this cell. A `.gif` uses Pillow, which
matplotlib always has; an `.mp4` needs `ffmpeg` on the PATH.

In [6]:
if SAVE_PATH is None:
    print("SAVE_PATH is None: nothing saved.")
else:
    save_path = Path(SAVE_PATH)
    if save_path.suffix.lower() == ".gif":
        writer = PillowWriter(fps=SAVE_FPS)
    else:
        writer = FFMpegWriter(fps=SAVE_FPS)
    anim.save(str(save_path), writer=writer, dpi=SAVE_DPI)
    set_parameter(molecule, PARAMETER, start_value)  # saving draws every frame again
    print(f"Saved: {save_path.resolve()}")

SAVE_PATH is None: nothing saved.


## 7. Another parameter: column density

The same function steps any other parameter. Here CO goes from $10^{15}$ to
$10^{20}$ cm$^{-2}$ at 1000 K. While the lines are optically thin, the diagram is a straight
line that only moves up as the column density grows, and the fit gives back the model's
temperature (about 1000 K at $10^{15}$ cm$^{-2}$). Once the strong low-$E_u$ lines become optically thick
they stop rising, the diagram curves, and the fitted $T_\mathrm{rot}$ climbs above the
model's 1000 K. With `SIZE_BY = "tau"` the points grow as the lines become optically thick,
so you can see which lines bend the diagram.

In [7]:
column_densities = np.logspace(15, 20, 21)
start_n_mol = molecule.n_mol
set_parameter(molecule, "temp", 1000.0)

anim_n, fig_n = make_population_diagram_animation(
    molecule, "n_mol", column_densities,
    marker_size=MARKER_SIZE, figsize=FIGSIZE, interval=FRAME_INTERVAL_MS,
    size_by=SIZE_BY, size_log=SIZE_LOG, size_limits=SIZE_LIMITS, size_range=SIZE_RANGE,
    x_axis=X_AXIS, y_axis=Y_AXIS, x_log=X_LOG, y_log=Y_LOG,
    theme=THEME,
)
with plt.rc_context({"savefig.dpi": INLINE_DPI, "animation.embed_limit": INLINE_MAX_MB}):
    html_n = anim_n.to_jshtml()
plt.close(fig_n)

set_parameter(molecule, "n_mol", start_n_mol)
set_parameter(molecule, PARAMETER, start_value)
display(HTML(html_n))

## Notes

- **Speed:** each frame recalculates the intensities and redraws the diagram, about 0.15 s
  for CO. For a long animation of a large line list, use fewer `VALUES` or
  `FIXED_LIMITS = False` (which skips the pass that finds the limits).
- **Size of the inline animation:** `to_jshtml()` embeds every frame in the notebook, at
  `INLINE_DPI` (iSLAT's 300 dpi export default would make them about 9 times larger).
  Matplotlib stops at `matplotlib.rcParams["animation.embed_limit"]` (20 MB by default);
  raise it for long animations, or save to a file instead.
- **Valid lines:** the fit, like the GUI, uses the lines whose flux is above 1 % of the
  strongest line's (`valid_mask` in `PopulationDiagramPlot.component_data`).